In [31]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [32]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [33]:
data = pd.read_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/ham_data.csv')

In [34]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [36]:
data[data["yesil_skor_notu"].isna()].sample(10)

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
6622,https://world.openfoodfacts.org/product/007680...,7.680853e+10,PROTEIN+ ROTINI – Barilla,NaN,NaN,barilla,"Plant-based foods and beverages, ,, Plant-base...","Vegetarian, ,, No GMOs, ,, Vegan, ,, Non GMO p...",NaN,NaN,...,0.0,0.0,0.0,4.0,2.0,0.0,PROTEIN+ ROTINI,None,"[Plant-based foods and beverages, Plant-based ...","[Vegetarian, No GMOs, Vegan, Non GMO project, ..."
3250,https://world.openfoodfacts.org/product/871890...,8.718908e+12,AH Terra Plantaardig 100% pindakaas naturel – ...,350.0,NaN,albert heijn terra,"Ontbijtgranen, ,, Pindakaas, ,, Zoet beleg, ,,...","Vegetarian, ,, Vegan, ,, Nutriscore, ,, Nutris...",NaN,NaN,...,1.0,6.0,0.0,NaN,5.0,0.0,AH Terra Plantaardig 100% pindakaas naturel,g,"[Ontbijtgranen, Pindakaas, Zoet beleg, Beleg, ...","[Vegetarian, Vegan, Nutriscore, Nutriscore Gra..."
63765,https://world.openfoodfacts.org/product/325039...,3.250392e+12,LS Le Mathurin L'élancé 200g – Monique ranou –...,200.0,NaN,monique ranou,"Meats and their products, ,, Meats, ,, Prepare...","French meat, ,, French pork, ,, Nutriscore",NaN,NaN,...,0.0,4.0,20.0,NaN,0.0,0.0,LS Le Mathurin L'élancé 200g,g,"[Meats and their products, Meats, Prepared mea...","[French meat, French pork, Nutriscore]"
18920,https://world.openfoodfacts.org/product/008304...,8.304645e+10,Spring Water – Ice Mountain – 6,6.0,Pet-bottle,ice mountain,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Spring Water,NaN,"[Beverages and beverages preparations, Beverag...",[]
25372,https://world.openfoodfacts.org/product/008523...,8.523905e+10,Spanish Style Rice With Tomatoes & Peppers – G...,NaN,NaN,good & gather,"Meals, ,, Microwave meals, ,, Pots, ,, Rice in...",No artificial flavors,NaN,NaN,...,0.0,0.0,2.0,0.0,0.0,0.0,Spanish Style Rice With Tomatoes & Peppers,None,"[Meals, Microwave meals, Pots, Rice in pots]",[No artificial flavors]
19254,https://world.openfoodfacts.org/product/899589...,8.995899e+12,Wonton Noodles – Bali Kitchen – 200g,200.0,NaN,bali kitchen,"Plant-based foods and beverages, ,, Plant-base...",NaN,NaN,NaN,...,0.0,0.0,5.0,7.0,1.0,0.0,Wonton Noodles,g,"[Plant-based foods and beverages, Plant-based ...",[]
41544,https://world.openfoodfacts.org/product/941488...,4.008165e+12,La délicieuse – 135g,135.0,NaN,la délicieuse,"Spreads, ,, Salted spreads","Organic, ,, EU Organic, ,, Non-EU Agriculture,...",NaN,NaN,...,0.0,2.0,4.0,2.0,0.0,0.0,La délicieuse,g,"[Spreads, Salted spreads]","[Organic, EU Organic, Non-EU Agriculture, BE-B..."
39967,https://world.openfoodfacts.org/product/505969...,5.059697e+12,Orange Quadruple strength – Tesco – 1.5l,1.5,NaN,tesco,"Beverages and beverages preparations, ,, Plant...",NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Orange Quadruple strength,l,"[Beverages and beverages preparations, Plant-b...",[]
58078,https://world.openfoodfacts.org/product/890149...,8.901491e+12,Masala munch,NaN,NaN,NaN,"Snacks, ,, Salty snacks, ,, Appetizers",NaN,NaN,NaN,...,0.0,10.0,11.0,NaN,0.0,0.0,Masala munch,None,"[Snacks, Salty snacks, Appetizers]",[]
32382,https://world.openfoodfacts.org/product/500016...,5.000170e+12,Wood-fired Mushroom & Truffle Sourdough Pizza ...,NaN,NaN,waitrose,"Meals, ,, Pizzas pies and quiches, ,, Pizzas",NaN,NaN,NaN,...,0.0,3.0,4.0,3.0,0.0,0.0,Wood-fired Mushroom & Truffle Sourdough Pizza,None,"[Meals, Pizzas pies and quiches, Pizzas]",[]


In [37]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [38]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001434
urun_adi                         0.002868
miktar                          19.241029
ambalaj                         58.770043
markalar                         3.513754
kategoriler                      0.002868
etiketler                       30.545277
mensei                          76.027594
uretim_yerleri                  80.076299
satildigi_ulkeler                0.108998
icerik_metni                    12.884720
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   13.198807
nutriscore_notu                  0.103261
nova_grubu                      15.433267
yesil_skor_notu                 27.038694
palmiye_yagi_icermez            19.154978
vejetaryen                      22.860913
vegan_durumu                    12.669592
yag_seviyesi                     2.331985
doymus_yag_seviyesi              3.355993
seker_seviyesi                   2

In [39]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","etiketler","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)


In [40]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [41]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (69726): ['1.020304050607081e+17', '1000029852900.0', '10001219.0', '10001295.0', '10001356.0', '10001400.0', '10001404.0', '10001691.0', '10001707.0', '10001875.0'] ...
miktar (1176): ['0.0', '0.03', '0.042', '0.045', '0.046', '0.05', '0.055', '0.06', '0.065', '0.07'] ...
markalar (13209): ['"tradition culinaire"', "'z bregov", '(sans marque)', '07x netto 03.25', '1 2 3 fruits', '1 attimo in forma', '1 l', '1 x auer 01.25', '1%', '1-2-3'] ...
alerjenler (1942): ["['Acesulfame-potassium']", "['Apple', 'Banana', 'Celery', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Nuts']", "['Apple', 'Banana', 'Gluten', 'Sulphur dioxide and sulphites']", "['Apple', 'Banana', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Kiwi', 'Orange', 'Peach']", "['Apple', 'Banana', 'Milk']", "['Apple', 'Banana', 'Orange', 'Peach']", "['Apple', 'Banana', 'Orange', 'Sulphur dioxide and sulphites']"] ...
eser_miktarlar (2547): ["['07

In [42]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     42772
False    13598
Name: count, dtype: int64

In [43]:
data.isnull().mean() * 100

barkod                         0.001434
miktar                        19.241029
markalar                       3.513754
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 13.198807
nutriscore_notu                0.103261
nova_grubu                    15.433267
yesil_skor_notu               27.038694
palmiye_yagi_icermez          19.154978
vejetaryen                    22.860913
vegan_durumu                  12.669592
enerji_kcal                    1.223360
yag_g                          1.236268
doymus_yag_g                   2.309038
karbonhidrat_g                 1.330924
seker_g                        1.699510
lif_g                         29.167025
protein_g                      1.270688
tuz_g                          1.085678
alkol_yuzde                   95.133809
meyve_sebze_baklagil_yuzde    67.912974
birim                         21.181482
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [44]:
data.to_csv('/Users/oguzhanerbil/Documents/Repolarım/food-health-predictor/data/data.csv', index=False)

In [45]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'alkol_yuzde', 'meyve_sebze_baklagil_yuzde',
       'birim', 'kategori_listesi', 'etiketler_listesi'],
      dtype='object')